# 02 - Preprocesamiento de los datasets

En este notebook se ejecuta el preprocesamiento de los cuatro datasets del proyecto de cribado de TEA:

- `adults`
- `combined`
- `toddlers`
- `adolescents`

El objetivo es transformar los datos crudos en conjuntos listos para modelado, aplicando de forma homogénea:

- limpieza de valores problemáticos,
- codificación de la variable objetivo,
- eliminación de columnas no aptas para entrenamiento,
- separación en entrenamiento y test,
- transformación de variables numéricas y categóricas,
- y guardado de los resultados procesados.

La lógica principal del preprocesamiento está centralizada en:

```python
src/preprocessing.py

In [8]:
import sys
from pathlib import Path

import pandas as pd
from scipy.io import arff

# Ruta raíz del proyecto
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

# Directorios de datos
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Módulo de preprocesamiento
import src.preprocessing as prep

In [9]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: C:\Users\adria\Documents\bootcamp\tesTEA
RAW_DIR: C:\Users\adria\Documents\bootcamp\tesTEA\data\raw
PROCESSED_DIR: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed


## 1. Carga de los datasets originales

Se cargan los cuatro datasets del proyecto desde la carpeta `data/raw`.

Cada uno de ellos será procesado con la configuración específica definida en `src/preprocessing.py`.  
Aunque los datasets no tienen exactamente la misma estructura, el módulo centraliza su configuración para aplicar una lógica de preprocesamiento homogénea.

In [ ]:
ADULTS_PATH = RAW_DIR / "autism_screening.csv"
COMBINED_PATH = RAW_DIR / "Autism_Screening_Data_Combined.csv"
TODDLERS_PATH = RAW_DIR / "Toddler Autism dataset July 2018.csv"
ADOLESCENTS_PATH = RAW_DIR / "Autism-Adolescent-Data.arff"

# Carga de datasets CSV
adults_df = prep.load_dataset(ADULTS_PATH)
combined_df = prep.load_dataset(COMBINED_PATH)
toddlers_df = prep.load_dataset(TODDLERS_PATH)

# Carga del dataset adolescents desde ARFF
adolescents_data, adolescents_meta = arff.loadarff(ADOLESCENTS_PATH)
adolescents_df = pd.DataFrame(adolescents_data)

# Decodificar columnas tipo bytes a string
adolescents_df = prep.decode_arff_bytes(adolescents_df)

## 2. Resumen inicial de los datasets

Antes de ejecutar el pipeline, se revisa de forma resumida la estructura de cada dataset:

- número de filas y columnas,
- variable objetivo,
- columnas que se eliminarán,
- variables numéricas, categóricas y AQ configuradas en el módulo de preprocesamiento.

Este resumen se obtiene con la función `get_dataset_summary()` del propio módulo, para asegurar que el notebook refleja exactamente la configuración real del pipeline.

In [11]:
adults_summary = prep.get_dataset_summary(adults_df, "adults")
combined_summary = prep.get_dataset_summary(combined_df, "combined")
toddlers_summary = prep.get_dataset_summary(toddlers_df, "toddlers")
adolescents_summary = prep.get_dataset_summary(adolescents_df, "adolescents")

In [12]:
print("Resumen adults:")
for k, v in adults_summary.items():
    print(f"{k}: {v}")

print("\n" + "="*70 + "\n")

print("Resumen combined:")
for k, v in combined_summary.items():
    print(f"{k}: {v}")

print("\n" + "="*70 + "\n")

print("Resumen toddlers:")
for k, v in toddlers_summary.items():
    print(f"{k}: {v}")

print("\n" + "="*70 + "\n")

print("Resumen adolescents:")
for k, v in adolescents_summary.items():
    print(f"{k}: {v}")

Resumen adults:
dataset: adults
n_rows: 704
n_cols: 21
target: Class/ASD
drop_cols: ['result', 'age_desc', 'score_total']
numeric_cols: ['age']
categorical_cols: ['gender', 'ethnicity', 'jundice', 'austim', 'contry_of_res', 'used_app_before', 'relation']
aq_cols: ['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score']
group_rare_cols: ['ethnicity', 'contry_of_res', 'relation']
rare_min_freq: 5


Resumen combined:
dataset: combined
n_rows: 6075
n_cols: 15
target: Class
drop_cols: ['score_total']
numeric_cols: ['Age']
categorical_cols: ['Sex', 'Jauundice', 'Family_ASD']
aq_cols: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10']
group_rare_cols: []
rare_min_freq: 5


Resumen toddlers:
dataset: toddlers
n_rows: 1054
n_cols: 19
target: Class/ASD Traits 
drop_cols: ['Qchat-10-Score', 'Case_No', 'score_total']
numeric_cols: ['Age_Mons']
categorical_cols: ['Sex', 'Ethnicity', 'Jaundice', 'Family_mem_with_ASD', 'Who c

## 3. Preprocesamiento del dataset `adults`

Para el dataset `adults`, el pipeline aplica los siguientes pasos:

1. Limpieza de valores `"?"` y conversión a `NaN`.
2. Codificación de la variable objetivo `Class/ASD` a formato binario.
3. Eliminación de columnas no utilizadas en modelado.
4. Separación entre variables predictoras y target.
5. División en entrenamiento y test con estratificación.
6. Aplicación del preprocesador:
   - numéricas → imputación con mediana + `StandardScaler`
   - categóricas → imputación con `"unknown"` + `OneHotEncoder`
   - variables AQ → se conservan mediante `passthrough`
7. Guardado del resultado procesado y del preprocessor entrenado.

In [13]:
adults_train, adults_test, adults_preprocessor = prep.preprocess_and_save_dataset(
    df=adults_df,
    dataset_name="adults",
    output_dir=PROCESSED_DIR
)

Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\adults_train.csv
Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\adults_test.csv
Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\adults_preprocessor.pkl


In [14]:
print("Adults train shape:", adults_train.shape)
print("Adults test shape:", adults_test.shape)

print("\nNulos en adults_train:", adults_train.isna().sum().sum())
print("Nulos en adults_test:", adults_test.isna().sum().sum())

print("\nDistribución target adults train:")
print(adults_train["target"].value_counts(dropna=False))

print("\nDistribución target adults test:")
print(adults_test["target"].value_counts(dropna=False))

Adults train shape: (563, 55)
Adults test shape: (141, 55)

Nulos en adults_train: 0
Nulos en adults_test: 0

Distribución target adults train:
target
0    412
1    151
Name: count, dtype: int64

Distribución target adults test:
target
0    103
1     38
Name: count, dtype: int64


## 4. Preprocesamiento del dataset `combined`

En el caso del dataset `combined`, el pipeline aplica la misma lógica general adaptada a su estructura específica:

- target: `Class`
- variable numérica principal: `Age`
- variables categóricas: `Sex`, `Jauundice`, `Family_ASD`
- variables AQ: `A1`–`A10`

El objetivo es dejar el dataset en un formato homogéneo respecto al resto para facilitar la comparación de modelos en la fase posterior.

In [15]:
combined_train, combined_test, combined_preprocessor = prep.preprocess_and_save_dataset(
    df=combined_df,
    dataset_name="combined",
    output_dir=PROCESSED_DIR
)

Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\combined_train.csv
Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\combined_test.csv
Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\combined_preprocessor.pkl


In [16]:
print("Combined train shape:", combined_train.shape)
print("Combined test shape:", combined_test.shape)

print("\nNulos en combined_train:", combined_train.isna().sum().sum())
print("Nulos en combined_test:", combined_test.isna().sum().sum())

print("\nDistribución target combined train:")
print(combined_train["target"].value_counts(dropna=False))

print("\nDistribución target combined test:")
print(combined_test["target"].value_counts(dropna=False))

Combined train shape: (4860, 18)
Combined test shape: (1215, 18)

Nulos en combined_train: 0
Nulos en combined_test: 0

Distribución target combined train:
target
0    3417
1    1443
Name: count, dtype: int64

Distribución target combined test:
target
0    854
1    361
Name: count, dtype: int64


## 5. Preprocesamiento del dataset `toddlers`

El dataset `toddlers` presenta una estructura algo más rica en variables categóricas y columnas auxiliares.  
Durante el preprocesamiento se eliminan variables que no deben entrar al modelo, como por ejemplo:

- `Case_No`
- `Qchat-10-Score`

Además, se transforman variables como:

- `Sex`
- `Ethnicity`
- `Jaundice`
- `Family_mem_with_ASD`
- `Who completed the test`

El resultado final vuelve a ser un conjunto `train` y `test` con todas las variables listas para modelado.

In [17]:
toddlers_train, toddlers_test, toddlers_preprocessor = prep.preprocess_and_save_dataset(
    df=toddlers_df,
    dataset_name="toddlers",
    output_dir=PROCESSED_DIR
)

Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\toddlers_train.csv
Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\toddlers_test.csv
Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\toddlers_preprocessor.pkl


In [18]:
print("Toddlers train shape:", toddlers_train.shape)
print("Toddlers test shape:", toddlers_test.shape)

print("\nNulos en toddlers_train:", toddlers_train.isna().sum().sum())
print("Nulos en toddlers_test:", toddlers_test.isna().sum().sum())

print("\nDistribución target toddlers train:")
print(toddlers_train["target"].value_counts(dropna=False))

print("\nDistribución target toddlers test:")
print(toddlers_test["target"].value_counts(dropna=False))

Toddlers train shape: (843, 33)
Toddlers test shape: (211, 33)

Nulos en toddlers_train: 0
Nulos en toddlers_test: 0

Distribución target toddlers train:
target
1    582
0    261
Name: count, dtype: int64

Distribución target toddlers test:
target
1    146
0     65
Name: count, dtype: int64


## 6. Preprocesamiento del dataset `adolescents`

El dataset `adolescents` amplía el análisis del proyecto incorporando un grupo de edad intermedio entre toddlers y adults.

Su estructura es muy similar a la del dataset `adults`, ya que contiene:

- preguntas AQ (`A1_Score`–`A10_Score`),
- una variable numérica de edad (`age`),
- variables categóricas como `gender`, `ethnicity`, `jundice`, `austim`, `contry_of_res`, `used_app_before` y `relation`,
- y la variable objetivo `Class/ASD`.

Además, incluye columnas auxiliares como `result` y `age_desc`, que no se utilizarán como variables predictoras durante el modelado.

Dado que este dataset procede de un archivo `.arff`, antes de aplicar el pipeline se realiza una conversión a `DataFrame` y una decodificación de columnas en formato bytes para poder integrarlo en el mismo flujo de preprocesamiento que el resto de datasets.

In [19]:
adolescents_train, adolescents_test, adolescents_preprocessor = prep.preprocess_and_save_dataset(
    df=adolescents_df,
    dataset_name="adolescents",
    output_dir=PROCESSED_DIR
)

Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\adolescents_train.csv
Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\adolescents_test.csv
Guardado: C:\Users\adria\Documents\bootcamp\tesTEA\data\processed\adolescents_preprocessor.pkl


In [20]:
print("Adolescents train shape:", adolescents_train.shape)
print("Adolescents test shape:", adolescents_test.shape)

print("\nNulos en adolescents_train:", adolescents_train.isna().sum().sum())
print("Nulos en adolescents_test:", adolescents_test.isna().sum().sum())

print("\nDistribución target adolescents train:")
print(adolescents_train["target"].value_counts(dropna=False))

print("\nDistribución target adolescents test:")
print(adolescents_test["target"].value_counts(dropna=False))

Adolescents train shape: (83, 29)
Adolescents test shape: (21, 29)

Nulos en adolescents_train: 0
Nulos en adolescents_test: 0

Distribución target adolescents train:
target
1    50
0    33
Name: count, dtype: int64

Distribución target adolescents test:
target
1    13
0     8
Name: count, dtype: int64


## 7. Archivos generados por el pipeline

La función `preprocess_and_save_dataset()` no solo devuelve los `DataFrame` procesados, sino que además guarda en `data/processed/`:

### Para cada dataset:
- `*_train.csv`
- `*_test.csv`
- `*_preprocessor.pkl`

Los archivos `.csv` serán utilizados como entrada para la fase de modelado, mientras que los archivos `.pkl` permiten reutilizar exactamente el mismo preprocesador entrenado sin tener que reconstruirlo desde cero.

In [21]:
processed_files = sorted([p.name for p in PROCESSED_DIR.iterdir() if p.is_file()])

print("Archivos generados en data/processed:")
for file in processed_files:
    print("-", file)

Archivos generados en data/processed:
- .gitkeep
- adolescents_preprocessor.pkl
- adolescents_test.csv
- adolescents_train.csv
- adults_preprocessor.pkl
- adults_test.csv
- adults_train.csv
- combined_preprocessor.pkl
- combined_test.csv
- combined_train.csv
- toddlers_preprocessor.pkl
- toddlers_test.csv
- toddlers_train.csv


## 7.1 Observaciones tras el preprocesamiento

Los resultados obtenidos muestran que el pipeline se ha aplicado correctamente sobre los cuatro datasets:

- en todos los casos se generan conjuntos de entrenamiento y test sin valores nulos,
- la variable objetivo queda codificada de forma homogénea como `target`,
- y el número de columnas resultante varía entre datasets en función de su estructura original y del número de categorías presentes en las variables categóricas tras la codificación one-hot.

En particular:
- `combined` genera un número menor de variables transformadas porque su estructura es más simple y contiene menos variables categóricas,
- `toddlers` y `adolescents` producen más columnas debido a la mayor diversidad de categorías en variables como etnia, relación familiar o persona que completa el test,
- y `adults` también amplía notablemente el número de columnas tras la codificación de variables categóricas como etnia, país o relación.

## 8. Conclusiones del preprocesamiento

Tras ejecutar este notebook:

- los cuatro datasets del proyecto han sido preprocesados con una lógica homogénea,
- las variables objetivo se han codificado a formato binario,
- las variables numéricas y categóricas se han transformado para modelado,
- no quedan valores nulos en los conjuntos resultantes,
- se han generado los archivos `train` y `test` de cada dataset,
- y además se ha guardado el `preprocessor` entrenado de cada uno en formato `.pkl`.

La lógica completa del pipeline queda centralizada en `src/preprocessing.py`, mientras que este notebook actúa como registro reproducible de su ejecución sobre los datasets del proyecto.